# Project for Deep learning course

In [1]:
# importing dependencies and downloading dataset

!uv sync

import datasets as d

dataset = d.load_dataset("fancyzhx/amazon_polarity")

Resolved 275 packages in 15ms
Prepared 3 packages in 313ms                                                 sentence-transformers ------------------------------ 701.78 KiB/742.88 KiB      sentence-transformers ------------------------------ 621.78 KiB/742.88 KiB      
Installed 3 packages in 130ms=6.2.0                         
 + colorful==0.5.8
 + prettyprinter==0.18.0
 + sentence-transformers==6.2.0


In [2]:
dataset

DatasetDict({
    train: Dataset({
        features: ['label', 'title', 'content'],
        num_rows: 3600000
    })
    test: Dataset({
        features: ['label', 'title', 'content'],
        num_rows: 400000
    })
})

Downloading the generative model:

In [3]:
from transformers import pipeline

MODEL_NAME = 'HuggingFaceTB/SmolLM-135M-Instruct'

pipe = pipeline('text-generation', 
                model=MODEL_NAME, 
                device_map='auto', 
                dtype='auto')

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/186 [00:00<?, ?B/s]

In [4]:
prompt =    'Based on these examples, classify the last text positive or negative: ' \
            'If youve played the game, you know how divine the music is! Every single song tells a story of the game, its that good! The greatest songs are without a doubt, Chrono Cross: Times Scar, Magical Dreamers: The Wind, The Stars, and the Sea and Radical Dreamers: Unstolen Jewel. (Translation varies) This music is perfect if you ask me, the best it can be. Yasunori Mitsuda just poured his heart on and wrote it down on paper. => positive;' \
            'This was a easy to read book that made me want to keep reading on and on, not easy to put down.It left me wanting to read the follow on, which I hope is coming soon. I used to read a lot but have gotten away from it. This book made me want to read again. Very enjoyable. => positive;' \
            'Its glaringly obvious that all of the glowing reviews have been written by the same person, perhaps the author herself. They all have the same misspellings and poor sentence structure that is featured in the book. Who made Veronica Haddon think she is an author? => negative;' \
            'sizes are much smaller than what is recomended in the chart. I tried to put it and sheer it!. I guess you should not buy this item in the internet..it is better to go to the store and check it => ;'


In [5]:
print(pipe(prompt, max_length=25, max_new_tokens=512))

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=512) and `max_length`(=25) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': 'Based on these examples, classify the last text positive or negative: If youve played the game, you know how divine the music is! Every single song tells a story of the game, its that good! The greatest songs are without a doubt, Chrono Cross: Times Scar, Magical Dreamers: The Wind, The Stars, and the Sea and Radical Dreamers: Unstolen Jewel. (Translation varies) This music is perfect if you ask me, the best it can be. Yasunori Mitsuda just poured his heart on and wrote it down on paper. => positive;This was a easy to read book that made me want to keep reading on and on, not easy to put down.It left me wanting to read the follow on, which I hope is coming soon. I used to read a lot but have gotten away from it. This book made me want to read again. Very enjoyable. => positive;Its glaringly obvious that all of the glowing reviews have been written by the same person, perhaps the author herself. They all have the same misspellings and poor sentence structure that is

## Hypertuning model

dataset prep

In [6]:
from pprint import PrettyPrinter
pprint = PrettyPrinter(compact=True).pprint

dataset = dataset.shuffle()
pprint(dataset["train"][0])

{'content': 'Please even extended edition that the original boring version? '
            'please... this is clearly a comercial move toward the theatrical '
            'exhibition. Who care about an extended edition of a already very '
            "long and extended movie? I hope there's no going to be a fourth "
            'movie of this series.Thank you.',
 'label': 0,
 'title': 'Even more extended version?'}


In [ ]:
import transformers

model_name_gen = "HuggingFaceTB/SmolLM-135M-Instruct"
model_name_bi = "bert-base-cased"
tokenizer_gen = transformers.AutoTokenizer.from_pretrained(model_name_gen)
tokenizer_bi = transformers.AutoTokenizer.from_pretrained(model_name_bi)

In [ ]:
# Define a simple function that applies the tokenizer
def tokenize(example):
    return tokenizer_gen(
        example["content"],
        max_length=128,
        truncation=True,
    )

# Apply the tokenizer to the whole dataset using .map()
dataset_gen = dataset.map(tokenize)

Map:   0%|          | 0/3600000 [00:00<?, ? examples/s]

Map:   0%|          | 0/400000 [00:00<?, ? examples/s]

In [ ]:
# Define a simple function that applies the tokenizer
def tokenize(example):
    return tokenizer_bi(
        example["content"],
        max_length=128,
        truncation=True,
    )

# Apply the tokenizer to the whole dataset using .map()
dataset_bi = dataset.map(tokenize)

### Model prep

In [11]:
import torch


# This gives a new name to the config class, just for convenience
BasicConfig = transformers.PretrainedConfig


# This is the model
class SimpleCNN(transformers.PreTrainedModel):

    config_class = BasicConfig

    # In the initialization method, one instantiates the layers
    # these will be the parameters of the model
    def __init__(self, config):
        super().__init__(config)
        # Embedding layer: vocab size x embedding dim
        self.embeddings = torch.nn.Embedding(
            num_embeddings=config.vocab_size,
            embedding_dim=config.embedding_dim
        )
        # Convolution layer
        self.convolution = torch.nn.Conv1d(
            config.embedding_dim,
            config.num_filters,
            config.filter_size,
            padding=1
        )
        # Activation function following convolution
        self.activation = torch.nn.ReLU()
        # Pooling layer: global max pooling, regardless of input length
        self.pooling_layer = torch.nn.AdaptiveMaxPool1d(
            output_size=1
        )
        # Output layer: num filters to output size
        self.output_layer = torch.nn.Linear(
            in_features=config.num_filters,
            out_features=config.num_labels
        )
        # Loss function: standard loss for classification
        self.loss = torch.nn.CrossEntropyLoss()

    def forward(self, input_ids, labels=None, attention_mask=None):
        #shape of input: [batch_size, maxlen]
        x = self.embeddings(input_ids)
        #shape of x: [batch_size, maxlen, embedding_dim]
        x = x.permute((0,2,1))
        #shape of x: [batch_size, embedding_dim, maxlen]
        x = self.convolution(x)
        #shape of x: [batch_size, filters, maxlen]
        x = self.activation(x)
        #shape of x: [batch_size, filters, maxlen]
        x = self.pooling_layer(x)
        #shape of x: [batch_size, filters, 1]
        x = x.flatten(start_dim=1)
        #shape of x: [batch_size, filters]
        output = self.output_layer(x)

        # Return value computed as in the MLP:
        if labels is not None:
            # We have labels, so we can calculate the loss
            return (self.loss(output,labels), output)
        else:
            # No labels, so just return the output
            return (output,)

In [ ]:
config_gen = BasicConfig(
    vocab_size = tokenizer_gen.vocab_size,
    num_labels = len(set(dataset['train']['label'])),
    embedding_dim = 64,
    filter_size = 3,
    num_filters = 10,
)

model_gen = SimpleCNN(config_gen)

In [ ]:
config_bi = BasicConfig(
    vocab_size = tokenizer_bi.vocab_size,
    num_labels = len(set(dataset['train']['label'])),
    embedding_dim = 64,
    filter_size = 3,
    num_filters = 10,
)

model_bi = SimpleCNN(config_bi)

In [13]:
# Set training arguments
trainer_args = transformers.TrainingArguments(
    "checkpoints",
    eval_strategy="steps",
    logging_strategy="steps",
    load_best_model_at_end=True,
    eval_steps=500,
    logging_steps=1000,
    learning_rate=0.005,
    per_device_train_batch_size=32,
    max_steps=5000,
    report_to="none",
)

In [ ]:
import evaluate

accuracy = evaluate.load("accuracy")


def compute_accuracy(outputs_and_labels):
    outputs, labels = outputs_and_labels
    predictions = outputs.argmax(axis=-1) #pick the index of the "winning" label
    return accuracy.compute(predictions=predictions, references=labels)

data_collator_gen = transformers.DataCollatorWithPadding(tokenizer_gen)
data_collator_bi = transformers.DataCollatorWithPadding(tokenizer_bi)

# Argument gives the number of steps of patience before early stopping
early_stopping = transformers.EarlyStoppingCallback(
    early_stopping_patience=5
)

In [16]:
from collections import defaultdict

class LogSavingCallback(transformers.TrainerCallback):
    def on_train_begin(self, *args, **kwargs):
        self.logs = defaultdict(list)
        self.training = True

    def on_train_end(self, *args, **kwargs):
        self.training = False

    def on_log(self, args, state, control, logs, model=None, **kwargs):
        if self.training:
            for k, v in logs.items():
                if k != "epoch" or v not in self.logs[k]:
                    self.logs[k].append(v)

training_logs = LogSavingCallback()

In [ ]:
trainer = transformers.Trainer(
    model=model_gen,
    args=trainer_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    compute_metrics=compute_accuracy,
    data_collator=data_collator_gen,
    callbacks=[early_stopping, training_logs]
)

trainer.train()

/home/lauris/Documents/School/V26-27/Deep-Learning/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss,Accuracy
500,No log,0.519800,0.738653
1000,0.527272,0.461759,0.781385
1500,0.527272,0.438468,0.797755


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/home/lauris/Documents/School/V26-27/Deep-Learning/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/home/lauris/Documents/School/V26-27/Deep-Learning/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/home/lauris/Documents/School/V26-27/Deep-Learning/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


KeyboardInterrupt: 

In [ ]:
trainer = transformers.Trainer(
    model=model_bi,
    args=trainer_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    compute_metrics=compute_accuracy,
    data_collator=data_collator_bi,
    callbacks=[early_stopping, training_logs]
)